In [51]:
import pandas as pd, scipy.stats as stats
url = "https://raw.githubusercontent.com/nkmwicz/worldcup2018data/refs/heads/main/cleaned_events_world_cup2018.csv"
df = pd.read_csv(url)
df.head()

,eventId,subEventName,tags,playerId,matchId,eventName,teamId,matchPeriod,eventSec,subEventId,id,x1,y1,x2,y2
0,8,Simple pass,['Accurate'],Mohammad Ibrahim Al Sahlawi,"Russia - Saudi Arabia, 5 - 0",Pass,Saudi Arabia,1H,1.656214,Simple pass,258612104,50,50,35.0,53.0
1,8,High pass,['Accurate'],Abdullah Ibrahim Otayf,"Russia - Saudi Arabia, 5 - 0",Pass,Saudi Arabia,1H,4.487814,High pass,258612106,35,53,75.0,19.0
2,1,Air duel,"['Won', 'Accurate']",Ilya Kutepov,"Russia - Saudi Arabia, 5 - 0",Duel,Russia,1H,5.937411,Air duel,258612077,25,81,37.0,83.0
3,1,Air duel,"['Lost', 'Not accurate']",Yasir Gharsan Al Shahrani,"Russia - Saudi Arabia, 5 - 0",Duel,Saudi Arabia,1H,6.406961,Air duel,258612112,75,19,63.0,17.0
4,8,Simple pass,['Accurate'],Salman Mohammed Al Faraj,"Russia - Saudi Arabia, 5 - 0",Pass,Saudi Arabia,1H,8.562167,Simple pass,258612110,63,17,71.0,15.0


In [52]:
df["eventName"].unique()

array(['Pass', 'Duel', 'Free Kick', 'Foul', 'Others on the ball', 'Shot',
       'Save attempt', 'Offside', 'Goalkeeper leaving line'], dtype=object)

In [53]:
# is each event a pass?
# is each event a goal?
# is each event a shot?
df["pass"] = (df["eventName"] == "Pass").astype(int)
df["shot"] = (df["eventName"] == "Shot").astype(int)
df["shot"].head()

0    0
1    0
2    0
3    0
4    0
Name: shot, dtype: int64

In [54]:
import ast 

df["tags"] = df["tags"].apply(ast.literal_eval)

In [55]:
tags = set([tag for taglist in df["tags"] for tag in taglist])
tags

{'Accurate',
 'Anticipated',
 'Anticipation',
 'Assist',
 'Blocked',
 'Counter attack',
 'Dangerous ball lost',
 'Direct',
 'Fairplay',
 'Feint',
 'Free space left',
 'Free space right',
 'Goal',
 'Head/body',
 'High',
 'Indirect',
 'Interception',
 'Key pass',
 'Left foot',
 'Lost',
 'Missed ball',
 'Neutral',
 'Not accurate',
 'Opportunity',
 'Own goal',
 'Position: Goal center',
 'Position: Goal center left',
 'Position: Goal center right',
 'Position: Goal high center',
 'Position: Goal high left',
 'Position: Goal high right',
 'Position: Goal low center',
 'Position: Goal low left',
 'Position: Goal low right',
 'Position: Out center left',
 'Position: Out center right',
 'Position: Out high center',
 'Position: Out high left',
 'Position: Out high right',
 'Position: Out low left',
 'Position: Out low right',
 'Position: Post center left',
 'Position: Post center right',
 'Position: Post high center',
 'Position: Post high left',
 'Position: Post high right',
 'Position: Post lo

In [56]:
df["goal"] = (
    (df["eventName"] == "Shot") & 
    (df["tags"].apply(lambda taglist: "Goal" in taglist))
    ).astype(int)


In [57]:
df.head()

,eventId,subEventName,tags,playerId,matchId,eventName,teamId,matchPeriod,eventSec,subEventId,id,x1,y1,x2,y2,pass,shot,goal
0,8,Simple pass,[Accurate],Mohammad Ibrahim Al Sahlawi,"Russia - Saudi Arabia, 5 - 0",Pass,Saudi Arabia,1H,1.656214,Simple pass,258612104,50,50,35.0,53.0,1,0,0
1,8,High pass,[Accurate],Abdullah Ibrahim Otayf,"Russia - Saudi Arabia, 5 - 0",Pass,Saudi Arabia,1H,4.487814,High pass,258612106,35,53,75.0,19.0,1,0,0
2,1,Air duel,"[Won, Accurate]",Ilya Kutepov,"Russia - Saudi Arabia, 5 - 0",Duel,Russia,1H,5.937411,Air duel,258612077,25,81,37.0,83.0,0,0,0
3,1,Air duel,"[Lost, Not accurate]",Yasir Gharsan Al Shahrani,"Russia - Saudi Arabia, 5 - 0",Duel,Saudi Arabia,1H,6.406961,Air duel,258612112,75,19,63.0,17.0,0,0,0
4,8,Simple pass,[Accurate],Salman Mohammed Al Faraj,"Russia - Saudi Arabia, 5 - 0",Pass,Saudi Arabia,1H,8.562167,Simple pass,258612110,63,17,71.0,15.0,1,0,0


In [58]:
from typing import List

def get_stats(columns: List[str], normalized: bool = False):
    """
    Args:
        columns (List[str]): columns to groupby
        normalized (bool): should we normalize shots, passes and goals to game?
    Returns:
        None
    """
    grp = df.groupby(columns).agg(
        passes=("pass", "sum"),
        shots=("shot", "sum"),
        goals=("goal", "sum"),
        matches=("matchId", "nunique")
    ).reset_index()
    if normalized: 
        grp["passes"] = grp["passes"] / grp["matches"]
        grp["shots"] = grp["shots"] / grp["matches"]
        grp["goals"] = grp["goals"] / grp["matches"]
    
    r, p = stats.pearsonr(grp["passes"], grp["shots"])
    r1, p1 = stats.pearsonr(grp["passes"], grp["goals"])
    
    print(f"agg by {'-'.join(columns)}{"norm" if normalized else None}: shots r:{r:.4f} p:{p:.4f} r2:{r**2:.4f}")
    print(f"agg by {'-'.join(columns)}{"norm" if normalized else None}: goals r:{r1:.4f} p:{p1:.4f} r2:{r1**2:.4f}")

## By Matches


In [59]:
get_stats(["matchId"])

agg by matchIdNone: shots r:0.1996 p:0.1138 r2:0.0398
agg by matchIdNone: goals r:0.0262 p:0.8369 r2:0.0007


## By Teams Total

In [60]:
get_stats(["teamId"])

agg by teamIdNone: shots r:0.8932 p:0.0000 r2:0.7979
agg by teamIdNone: goals r:0.8541 p:0.0000 r2:0.7295


## By Teams Normalized


In [61]:
get_stats(["teamId"], normalized=True)

agg by teamIdnorm: shots r:0.6667 p:0.0000 r2:0.4445
agg by teamIdnorm: goals r:0.4271 p:0.0148 r2:0.1824


## By Match in Teams

In [62]:
get_stats(["matchId", "teamId"])

agg by matchId-teamIdNone: shots r:0.5642 p:0.0000 r2:0.3184
agg by matchId-teamIdNone: goals r:0.1200 p:0.1772 r2:0.0144
